# Real-Time vs Batch Inference — AI Lab Instructor Notebook

*System Design · Expert*



5 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# Real-Time vs Batch Inference — Student Lab

Offline lab: compare scoring policies and measure freshness/cost tradeoffs.

## 0 — Simulated world (risk changes over time)

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


**Why this works.** # Real-Time vs Batch Inference — Instructor Lab

Offline lab: compare scoring policies and measure freshness/cost tradeoffs.

## 0 — Simulated world (risk changes over time)

## Task 2: 0 — Simulated world (risk changes over time)

## 0 — Simulated world (risk changes over time)

In [ ]:
N_USERS = 500
HOURS = 72
EFFECTIVE_WINDOW_H = 3
THRESH = 0.65

def simulate_risk(n_users: int, hours: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    base = rng.uniform(0.05, 0.25, size=(n_users,))
    risk = np.zeros((n_users, hours), dtype=np.float64)
    risk[:, 0] = base
    for t in range(1, hours):
        drift = 0.01 * rng.standard_normal((n_users,))
        risk[:, t] = np.clip(risk[:, t-1] + drift, 0.0, 1.0)
        spike_mask = rng.random((n_users,)) < 0.01
        if np.any(spike_mask):
            add = rng.uniform(0.3, 0.7, size=(int(spike_mask.sum()),))
            risk[spike_mask, t] = np.clip(risk[spike_mask, t] + add, 0.0, 1.0)
    return risk

risk = simulate_risk(N_USERS, HOURS, seed=0)
check('risk_shape', risk.shape == (N_USERS, HOURS))


In [ ]:
# Checks
check('risk_shape', risk.shape == (N_USERS, HOURS))

## Task 3: 1 — Spike events (threshold crossings)

## 1 — Spike events (threshold crossings)

In [ ]:
def find_spike_times_vectorized(risk, thresh):
    above = risk >= thresh

    # Already above threshold at hour 0
    users0 = np.where(above[:, 0])[0]
    spikes0 = [(u, 0) for u in users0]

    # False -> True transitions from hour 1 onward
    starts = above[:, 1:] & ~above[:, :-1]
    users, times = np.where(starts)

    spikes = spikes0 + list(zip(users, times + 1))
    return spikes

spikes = find_spike_times_vectorized(risk, THRESH)
print('n_spikes', len(spikes))
check('spikes_is_list', isinstance(spikes, list))

In [ ]:
# Checks
check('spikes_is_list', isinstance(spikes, list))

**Why this works.** Vectorized threshold-crossing detection avoids Python loops and identifies all False->True transitions efficiently.

## Task 4: 2 — Policies: realtime vs batch

## 2 — Policies: realtime vs batch

Implement two policies:
- `realtime`: score immediately when a spike event happens
- `batch`: score all users every `batch_every_h` hours

Return a summary dict with at least:
- `policy`
- `batch_every_h`
- `n_spikes`
- `caught`
- `catch_rate`
- `avg_delay_h`
- `score_calls`

Interpretation:
- lower `avg_delay_h` = fresher reaction
- lower `score_calls` = cheaper policy
- higher `catch_rate` = more useful detections within the effective window

In [ ]:
$f

In [ ]:
# Checks
check('policy_dict', isinstance(summary_rt, dict))
check('policy_keys', all(k in summary_rt for k in ['policy','n_spikes','caught','catch_rate','avg_delay_h','score_calls']))
check('realtime_full_catch', summary_rt['caught'] == summary_rt['n_spikes'])
check('batch_cost_higher', summary_b1['score_calls'] >= summary_rt['score_calls'])

## Task 5: 3 — Compare policies

## 3 — Compare policies

In [ ]:
results = []
results.append(run_policy(risk, spikes, policy="realtime"))
results.append(run_policy(risk, spikes, policy="batch", batch_every_h=1))
results.append(run_policy(risk, spikes, policy="batch", batch_every_h=6))
results.append(run_policy(risk, spikes, policy="batch", batch_every_h=24))

for r in results:
    print(r)
check('have4', len(results) == 4)
check('batch24_not_fresher_than_rt', results[3]['avg_delay_h'] >= results[0]['avg_delay_h'])
check('batch24_not_better_catch_than_rt', results[3]['catch_rate'] <= results[0]['catch_rate'] + 1e-12)
check('batch24_cheaper_than_hourly', results[3]['score_calls'] <= results[1]['score_calls'])

print('interpretation', 'realtime is freshest and catches all spikes immediately; less frequent batch scoring is cheaper but staler and misses more short-lived spikes.')

In [ ]:
# Checks
check('have4', len(results) == 4)
check('batch24_not_fresher_than_rt', results[3]['avg_delay_h'] >= results[0]['avg_delay_h'])
check('batch24_not_better_catch_than_rt', results[3]['catch_rate'] <= results[0]['catch_rate'] + 1e-12)
check('batch24_cheaper_than_hourly', results[3]['score_calls'] <= results[1]['score_calls'])